In [1]:
%pip install --quiet --upgrade google-cloud-bigquery google-cloud-bigquery-connection google-cloud-resource-manager pandas db-dtypes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 72.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.


In [2]:
from google.cloud import bigquery
from google.cloud import bigquery_connection_v1 as bq_connection
from google.cloud import resourcemanager_v3
from google.iam.v1 import iam_policy_pb2
import time

print("Libraries imported.")

Libraries imported.


In [3]:
import google.auth
_credentials, _detected_project = google.auth.default()

PROJECT_ID = _detected_project
LOCATION   = "US"

# --- BigQuery objects ---------------------------------------------------
DATASET_ID    = "aurora_bay"
CONNECTION_ID = "vertex_conn"           # BigQuery -> Vertex AI cloud-resource connection

RAW_TABLE        = f"{PROJECT_ID}.{DATASET_ID}.aurora_bay_faqs"
EMBEDDED_TABLE   = f"{PROJECT_ID}.{DATASET_ID}.aurora_bay_faqs_embedded"
EMBED_MODEL      = f"{PROJECT_ID}.{DATASET_ID}.embedding_model"
GEN_MODEL        = f"{PROJECT_ID}.{DATASET_ID}.gemini_model"

# --- Model endpoints -------------------------------
EMBED_MODEL_ENDPOINT = "text-embedding-005"
GEN_MODEL_ENDPOINT   = "gemini-2.5-flash"

# --- Source data --------------------------------------------------------
SOURCE_CSV = "gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv"

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
print(f"Project:  {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Dataset:  {DATASET_ID}")

Project:  qwiklabs-gcp-04-dc5ba3682c9e
Location: US
Dataset:  aurora_bay


In [4]:
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = LOCATION
# exists_ok=True makes re-runs safe.
dataset = client.create_dataset(dataset_ref, exists_ok=True)
print(f"Dataset ready: {dataset.full_dataset_id}")

Dataset ready: qwiklabs-gcp-04-dc5ba3682c9e:aurora_bay


In [5]:
load_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,           # header row
    autodetect=True,               # infer column names + types from the CSV
    write_disposition="WRITE_TRUNCATE",  # idempotent: reload replaces the table
)

load_job = client.load_table_from_uri(
    SOURCE_CSV, RAW_TABLE, job_config=load_config
)
load_job.result()  # wait for completion

raw_tbl = client.get_table(RAW_TABLE)
print(f"Loaded {raw_tbl.num_rows} rows into {RAW_TABLE}\n")
print("Schema:")
for f in raw_tbl.schema:
    print(f"  - {f.name} ({f.field_type})")

Loaded 50 rows into qwiklabs-gcp-04-dc5ba3682c9e.aurora_bay.aurora_bay_faqs

Schema:
  - string_field_0 (STRING)
  - string_field_1 (STRING)


In [6]:
# Preview the data
client.query(f"SELECT * FROM `{RAW_TABLE}` LIMIT 5").to_dataframe()

,string_field_0,string_field_1
0,When was Aurora Bay founded?,Aurora Bay was founded in 1901 by a group of f...
1,What is the population of Aurora Bay?,Aurora Bay has a population of approximately 3...
2,Where is the Aurora Bay Town Hall located?,The Town Hall is located at 100 Harbor View Ro...
3,Who is the current mayor of Aurora Bay?,"The current mayor is Linda Greenwood, elected ..."
4,What are the primary industries in Aurora Bay?,The primary industries include commercial fish...


In [9]:
# Detect the question/answer column names so the rest of the notebook is robust
# to the exact CSV header.
col_names = [f.name for f in raw_tbl.schema]
print(col_names)

['string_field_0', 'string_field_1']


In [10]:
QUESTION_COL = "string_field_0"
ANSWER_COL   = "string_field_1"

print(f"Question column: {QUESTION_COL!r}")
print(f"Answer column:   {ANSWER_COL!r}")
print(f"All columns:     {col_names}")

Question column: 'string_field_0'
Answer column:   'string_field_1'
All columns:     ['string_field_0', 'string_field_1']


In [11]:
## Create the BigQuery → Vertex AI connection (idempotent)
# A `CLOUD_RESOURCE` connection gives BigQuery a service-account identity it uses to call Vertex AI. We create it with the Python connection client (get-or-create), then read back its service-account email so we can grant it IAM in the next cell.

conn_client = bq_connection.ConnectionServiceClient()
parent = f"projects/{PROJECT_ID}/locations/{LOCATION.lower()}"
conn_path = f"{parent}/connections/{CONNECTION_ID}"

try:
    conn = conn_client.get_connection(name=conn_path)
    print(f"Connection already exists: {CONNECTION_ID}")
except Exception:
    conn = conn_client.create_connection(
        parent=parent,
        connection_id=CONNECTION_ID,
        connection=bq_connection.Connection(
            cloud_resource=bq_connection.CloudResourceProperties()
        ),
    )
    print(f"Created connection: {CONNECTION_ID}")

CONN_SERVICE_ACCOUNT = conn.cloud_resource.service_account_id
print(f"Connection service account: {CONN_SERVICE_ACCOUNT}")

Created connection: vertex_conn
Connection service account: bqcx-460143586717-j8jz@gcp-sa-bigquery-condel.iam.gserviceaccount.com


In [12]:
## Grant the connection's service account the Vertex AI User role
ROLE = "roles/aiplatform.user"
member = f"serviceAccount:{CONN_SERVICE_ACCOUNT}"

proj_client = resourcemanager_v3.ProjectsClient()
resource = f"projects/{PROJECT_ID}"

policy = proj_client.get_iam_policy(
    request=iam_policy_pb2.GetIamPolicyRequest(resource=resource)
)

binding = next((b for b in policy.bindings if b.role == ROLE), None)
if binding is None:
    binding = policy.bindings.add()
    binding.role = ROLE

if member not in binding.members:
    binding.members.append(member)
    proj_client.set_iam_policy(
        request=iam_policy_pb2.SetIamPolicyRequest(resource=resource, policy=policy)
    )
    print(f"Granted {ROLE} to {member}")
    # IAM propagation can lag; give it a moment before the remote model calls Vertex AI.
    print("Waiting 60s for IAM propagation...")
    time.sleep(60)
else:
    print(f"{member} already has {ROLE}")

Granted roles/aiplatform.user to serviceAccount:bqcx-460143586717-j8jz@gcp-sa-bigquery-condel.iam.gserviceaccount.com
Waiting 60s for IAM propagation...


In [13]:
## Create the remote models

# Two remote models, both backed by the connection above:
# - **Embedding model** → `text-embedding-005`, used by `ML.GENERATE_EMBEDDING`.
# - **Generation model** → Gemini, used by `AI.GENERATE_TEXT`.

def run_sql(sql, label=None):
    """Run a SQL statement and block until it finishes."""
    job = client.query(sql)
    job.result()
    if label:
        print(f"\u2713 {label}")
    return job

# The connection is referenced as `LOCATION.CONNECTION_ID` in DDL.
CONN_REF = f"{LOCATION.lower()}.{CONNECTION_ID}"

run_sql(f"""
CREATE OR REPLACE MODEL `{EMBED_MODEL}`
REMOTE WITH CONNECTION `{CONN_REF}`
OPTIONS (ENDPOINT = '{EMBED_MODEL_ENDPOINT}')
""", label=f"Embedding model created ({EMBED_MODEL_ENDPOINT})")

run_sql(f"""
CREATE OR REPLACE MODEL `{GEN_MODEL}`
REMOTE WITH CONNECTION `{CONN_REF}`
OPTIONS (ENDPOINT = '{GEN_MODEL_ENDPOINT}')
""", label=f"Generation model created ({GEN_MODEL_ENDPOINT})")

✓ Embedding model created (text-embedding-005)
✓ Generation model created (gemini-2.5-flash)


QueryJob<project=qwiklabs-gcp-04-dc5ba3682c9e, location=US, id=875f12d5-b9d3-4d19-af2f-cce1687e18da>

In [15]:
## Generate and store embeddings for every FAQ

# We embed a combined **"Question … Answer …"** string for each record so the
# vector captures both the question phrasing and the answer content — this improves
# retrieval recall. The result table keeps **all original fields** alongside the
# embedding vector, as the challenge requires.

# `ML.GENERATE_EMBEDDING` returns the vector in the `ml_generate_embedding_result` column.
run_sql(f"""
CREATE OR REPLACE TABLE `{EMBEDDED_TABLE}` AS
SELECT *
FROM ML.GENERATE_EMBEDDING(
  MODEL `{EMBED_MODEL}`,
  (
    SELECT
      *,
      CONCAT('Question: ', {QUESTION_COL}, '\\nAnswer: ', {ANSWER_COL}) AS content
    FROM `{RAW_TABLE}`
  ),
  STRUCT(TRUE AS flatten_json_output)
)
""", label="Embeddings generated and stored")

emb_tbl = client.get_table(EMBEDDED_TABLE)
print(f"\n{EMBEDDED_TABLE}: {emb_tbl.num_rows} rows")
print("Columns:", [f.name for f in emb_tbl.schema])

✓ Embeddings generated and stored

qwiklabs-gcp-04-dc5ba3682c9e.aurora_bay.aurora_bay_faqs_embedded: 50 rows
Columns: ['ml_generate_embedding_result', 'ml_generate_embedding_statistics', 'ml_generate_embedding_status', 'string_field_0', 'string_field_1', 'content']


In [16]:
# Sanity check: confirm we have a non-empty embedding vector per row.
client.query(f"""
SELECT
  {QUESTION_COL} AS question,
  ARRAY_LENGTH(ml_generate_embedding_result) AS embedding_dim
FROM `{EMBEDDED_TABLE}`
LIMIT 5
""").to_dataframe()

,question,embedding_dim
0,Who is the current mayor of Aurora Bay?,768
1,When was Aurora Bay founded?,768
2,What are the primary industries in Aurora Bay?,768
3,Does Aurora Bay have a public library?,768
4,Does Aurora Bay have a police department?,768


In [17]:
## The RAG chatbot

# The chatbot runs in three steps, all inside a single BigQuery query:

# 1. **Embed the user's question** with the same embedding model (`ML.GENERATE_EMBEDDING`).
# 2. **Retrieve** the most similar FAQs with `VECTOR_SEARCH` (cosine distance, top-K nearest neighbors).
# 3. **Generate** a grounded answer by passing the retrieved FAQ text + the question to Gemini via `AI.GENERATE_TEXT`.

# We split this into a `retrieve()` helper (so we can inspect what the vector search found) and an `ask()` function that adds the generation step.

def retrieve(question, top_k=5):
    """Step 1+2: embed the question and return the top_k most relevant FAQs."""
    sql = f"""
    SELECT
      base.{QUESTION_COL} AS question,
      base.{ANSWER_COL}   AS answer,
      distance
    FROM VECTOR_SEARCH(
      TABLE `{EMBEDDED_TABLE}`,
      'ml_generate_embedding_result',
      (
        SELECT ml_generate_embedding_result
        FROM ML.GENERATE_EMBEDDING(
          MODEL `{EMBED_MODEL}`,
          (SELECT @question AS content),
          STRUCT(TRUE AS flatten_json_output)
        )
      ),
      top_k => @top_k,
      distance_type => 'COSINE'
    )
    ORDER BY distance
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("question", "STRING", question),
            bigquery.ScalarQueryParameter("top_k", "INT64", top_k),
        ]
    )
    return client.query(sql, job_config=job_config).to_dataframe()

In [18]:
# Quick test of retrieval only — see which FAQs the vector search surfaces.
retrieve("How do I report a pothole?", top_k=3)

,question,answer,distance
0,How do I report a power outage?,Report power outages to the Aurora Bay Utiliti...,0.442556
1,How do I request a building permit?,Building permit applications can be obtained a...,0.447966
2,What are the winter road conditions usually like?,Road conditions can be icy and snowy from Nove...,0.471886


In [23]:
def ask(question, top_k=5, temperature=0.2, verbose=True):
    """Full RAG: retrieve relevant FAQs, then have Gemini answer grounded in them.

    Returns the generated answer string.
    """
    # Step 1+2: retrieve context
    ctx = retrieve(question, top_k=top_k)
    if ctx.empty:
        return "I couldn't find anything relevant in the Aurora Bay FAQs."

    context_block = "\n\n".join(
        f"Q: {r.question}\nA: {r.answer}" for r in ctx.itertuples()
    )

    if verbose:
        print("Retrieved context:")
        for r in ctx.itertuples():
            print(f"  • ({r.distance:.3f}) {r.question}")
        print()

    # Step 3: generation. Build a grounded prompt and call Gemini via
    # ML.GENERATE_TEXT, using the remote GEN_MODEL created earlier.
    prompt = f"""You are the helpful assistant for the town of Aurora Bay, Alaska.
Answer the user's question using ONLY the FAQ entries provided below.
If the answer is not contained in them, say you don't have that information
and suggest contacting the town office. Be concise and friendly.

FAQ context:
{context_block}

User question: {question}

Answer:"""

    # ML.GENERATE_TEXT is a table-valued function:
    #  - it takes a remote MODEL plus an input subquery with a `prompt` column,
    #  - its settings STRUCT must hold *literal constants* (so we inline
    #    temperature/max_output_tokens rather than passing them as parameters),
    #  - flatten_json_output=TRUE pulls the text into `ml_generate_text_llm_result`.
    # The prompt itself comes in via a query parameter (allowed in the data
    # subquery position), which keeps the FAQ text safely quoted.
    sql = f"""
    SELECT ml_generate_text_llm_result AS answer
    FROM ML.GENERATE_TEXT(
      MODEL `{GEN_MODEL}`,
      (SELECT @prompt AS prompt),
      STRUCT(
        {float(temperature)} AS temperature,
        1024 AS max_output_tokens,
        TRUE AS flatten_json_output
      )
    )
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("prompt", "STRING", prompt),
        ]
    )
    result = client.query(sql, job_config=job_config).to_dataframe()
    return result["answer"].iloc[0]

In [25]:
answer = ask("How do I contact the Aurora Bay Fire Department?")
print("\nAnswer:\n" + answer)

Retrieved context:
  • (0.118) How do I contact the Aurora Bay Fire Department?
  • (0.250) Does Aurora Bay have a police department?
  • (0.279) Is there a local hospital in Aurora Bay?
  • (0.284) How do I report a power outage?
  • (0.301) Where can I find information about local fishing regulations?


Answer:
The volunteer-based Aurora Bay Fire Department is reached at (907) 555-0123 for non-emergencies. For emergencies, always dial 911.


In [28]:
for q in [
    "Where is the Aurora Bay Town Hall located?",
    "When is the farmers market open?",
    "How do I pay my water bill?",
    "What is the population of Aurora Bay?",
    "How do I contact the Aurora Bay Fire Department?",
]:
    print("=" * 80)
    print("Q:", q)
    print("-" * 80)
    print(ask(q, verbose=False))
    print()

Q: Where is the Aurora Bay Town Hall located?
--------------------------------------------------------------------------------
The Aurora Bay Town Hall is located at 100 Harbor View Road, in the center of Aurora Bay, close to the main harbor.

Q: When is the farmers market open?
--------------------------------------------------------------------------------
I don't have that information. Please contact the town office.

Q: How do I pay my water bill?
--------------------------------------------------------------------------------
You can pay your water bill online, by mail, or in person at the Aurora Bay Utilities Department located within Town Hall.

Q: What is the population of Aurora Bay?
--------------------------------------------------------------------------------
Aurora Bay has a population of approximately 3,200 residents, although it can fluctuate seasonally due to temporary fishing and tourism workforces.

Q: How do I contact the Aurora Bay Fire Department?
----------------